# YOLO11 实例分割应用案例

本案例基于 MindSpore 适配实现的 `ultralytics`，演示 YOLO11 实例分割任务的训练、评估与推理完整流程。

In [1]:
import gc
import importlib
import os
import sys
import urllib.request
from pathlib import Path

import mindspore as ms

custom_ultralytics_parent = Path("/root/mindnlp/src/mindnlp")
if str(custom_ultralytics_parent) not in sys.path:
    sys.path.insert(0, str(custom_ultralytics_parent))

for module_name in list(sys.modules):
    if module_name == "ultralytics" or module_name.startswith("ultralytics."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import ultralytics
from ultralytics import YOLO

ms.set_context(mode=ms.PYNATIVE_MODE, device_target="Ascend")
package_dir = Path(ultralytics.__file__).resolve().parent
os.chdir(package_dir)
work_dir = Path.cwd() / "demo_inputs"
work_dir.mkdir(parents=True, exist_ok=True)

print("MindSpore version:", ms.__version__)
print("ultralytics package:", ultralytics.__file__)
print("package_dir:", package_dir)
print("cwd:", Path.cwd())

/root/.conda/envs/mindnlp_yolo/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/root/.conda/envs/mindnlp_yolo/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/root/.conda/envs/mindnlp_yolo/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/root/.conda/envs/mindnlp_yolo/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
[WARNING] ME(755:281473668799040,MainProcess):2026-04-24-20:47:28.600.

MindSpore version: 2.8.0
ultralytics package: /root/mindnlp/src/mindnlp/ultralytics/__init__.py
package_dir: /root/mindnlp/src/mindnlp/ultralytics
cwd: /root/mindnlp/src/mindnlp/ultralytics


## 1. 数据与测试图片准备

训练与评估默认使用 `cfg/datasets/coco128-seg.yaml`。推理阶段优先使用数据集中的样例图片；若当前环境下没有可用测试图片，则自动下载一张示例图片。

In [2]:
data_yaml = package_dir / "cfg/datasets/coco128-seg.yaml"
scratch_model_path = package_dir / "cfg/models/11/yolo11-seg.yaml"
finetune_model_path = package_dir / "yolo11n-seg.pt"

def resolve_source():
    candidates = [
        package_dir / "datasets/coco128-seg/images/train2017",
        package_dir / "datasets/coco128-seg/images/val2017",
    ]
    suffixes = {".jpg", ".jpeg", ".png", ".bmp"}
    for candidate in candidates:
        if candidate.is_file():
            return candidate
        if candidate.is_dir():
            files = sorted([p for p in candidate.rglob("*") if p.suffix.lower() in suffixes])
            if files:
                return files[0]

    image_path = work_dir / "segment_bus.jpg"
    if not image_path.exists():
        urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", image_path.as_posix())
        print("测试图片下载完成:", image_path)
    else:
        print("测试图片已存在:", image_path)
    return image_path

source_img = resolve_source()
print("数据配置:", data_yaml)
print("推理图片:", source_img)

数据配置: /root/mindnlp/src/mindnlp/ultralytics/cfg/datasets/coco128-seg.yaml
推理图片: /root/mindnlp/src/mindnlp/ultralytics/datasets/coco128-seg/images/train2017/000000000009.jpg


## 2. 模型训练

分割任务支持使用预训练权重进行微调，也支持使用模型配置文件从头开始训练。

In [3]:
# 微调
model = YOLO(finetune_model_path.as_posix())
# 从头开始训练
#model = YOLO(scratch_model_path.as_posix())

train_results = model.train(
    data=data_yaml.as_posix(),
    epochs=100,
    imgsz=640,
    batch_size=4,
    amp=False,
    val_interval=10,
    workers=8,
)

print("训练完成。")
print("best_fitness:", getattr(train_results, "best_fitness", None))
print("save_dir:", getattr(train_results, "save_dir", None))

[MindNLP YOLO] 发现已存在权重: /root/mindnlp/src/mindnlp/ultralytics/yolo11n-seg.ckpt，直接加载。
[MindNLP YOLO] 准备启动 segment 任务的训练...


2026-04-24 20:47:29,430 - INFO - 正在载入初始检查点权重: /root/mindnlp/src/mindnlp/ultralytics/yolo11n-seg.ckpt
2026-04-24 20:47:29,668 - INFO - ----------------------------------------
2026-04-24 20:47:29,669 - INFO - [权重匹配核查报告]
2026-04-24 20:47:29,670 - INFO - 待加载参数总量: 472
2026-04-24 20:47:29,670 - INFO - 网络未被初始化的参数数目: 0
2026-04-24 20:47:29,671 - INFO - ----------------------------------------


[INFO] 训练任务启动，总轮数: 1 epochs
Epoch [0/0] Step [0/32] | Total Loss: 7.6210 | Box: 1.5483 | Cls: 1.8655 | DFL: 1.5363 | Mask: 2.6710
Epoch [0/0] Step [10/32] | Total Loss: 14.9426 | Box: 3.1877 | Cls: 3.8860 | DFL: 3.2736 | Mask: 4.5954
Epoch [0/0] Step [20/32] | Total Loss: 7.7878 | Box: 2.0396 | Cls: 2.7479 | DFL: 2.0061 | Mask: 0.9941
Epoch [0/0] Step [30/32] | Total Loss: 8.7381 | Box: 2.1941 | Cls: 3.3331 | DFL: 2.3557 | Mask: 0.8552
训练完成。
best_fitness: 0.0
save_dir: runs/segment/train


## 3. 模型评估

训练完成后，在验证集上执行实例分割评估。

In [40]:
val_results = model.val(data=data_yaml.as_posix())
print("评估完成。")
print(val_results)

[MindNLP YOLO] 准备启动 segment 任务的验证...


Validating: 100%|██████████| 8/8 [02:17<00:00, 17.18s/it]
2026-04-24 18:49:19,613 - INFO - 推理测速: preprocess: 34.5ms | inference: 433.7ms | postprocess: 565.8ms


评估完成。
{'speed': {'preprocess': 34.50571186840534, 'inference': 433.7215945124626, 'postprocess': 565.8360552042723}, 'metrics/mAP50(B)': 9.325964017929409e-06, 'metrics/mAP50-95(B)': 1.6029701454275816e-06, 'metrics/mAP50(M)': 0.0, 'metrics/mAP50-95(M)': 0.0, 'fitness': 2.3752695326777644e-06}


## 4. 模型推理

推理结果会自动保存到输出目录，用户可直接查看可视化结果。

In [41]:
gc.collect()

predict_results = model(
    source=source_img.as_posix(),
    imgsz=640,
    conf=0.25,
    iou=0.45,
    save=True,
)

print("推理完成。")
if len(predict_results) > 0:
    print("推理结果保存目录:", getattr(predict_results[0], "save_dir", None))

2026-04-24 18:50:49,642 - INFO -  推理结果将保存至: /root/mindnlp/src/mindnlp/ultralytics/runs/detect/predict
2026-04-24 18:50:49,644 - INFO - 推理引擎启动，共探测到 1 份输入样本。


[MindNLP YOLO] 准备启动 segment 任务的推理...


2026-04-24 18:50:50,757 - INFO - 处理完成 [000000000009.jpg] | 前向推理: 903.8ms | 后处理: 105.1ms


推理完成。
推理结果保存目录: /root/mindnlp/src/mindnlp/ultralytics/runs/detect/predict
